# PI Advisor — Educational Notebook

**A step-by-step soil water balance (SWB) and irrigation scheduling tutorial**
based on FAO Irrigation & Drainage Paper 56 (Allen et al., 1998), driven by
**WaPOR v3** (FAO), **CHIRPS** rainfall and **ERA5-Land** climate, all
accessed through **Google Earth Engine**.

This notebook is a teaching version of the original Streamlit app
(`app.py`). The science is identical; the plumbing has been stripped so you
can run **one cell at a time** and inspect every input and output.

## What you will do

1. Authenticate to Google Earth Engine.
2. Upload a soil shapefile and a crop shapefile.
3. Choose the WaPOR resolution level (L1 = 300 m, L2 = 100 m, L3 = 30 m).
4. Pull historical satellite + climate data for your area.
5. Build a climatological forecast for any days that lie in the future.
6. Configure crop and soil parameters per zone.
7. Run the FAO-56 dual-Kc soil water balance.
8. Inspect a recommended irrigation schedule, **accept / reject / modify**
   individual events, and watch the soil moisture deficit recompute.
9. Export results.

## Required attribute columns in your shapefiles

| File          | Required column |
| ------------- | --------------- |
| Soil shapefile | `soil_zone`     |
| Crop shapefile | `crop_zone`     |

These mirror the original app's contract.


## 1. Install dependencies (Colab)

Skip this cell if you have already installed everything.
On a fresh Colab runtime this takes ~1 minute.

In [ ]:
# Run once per Colab session.
# In VS Code, install these in your environment first and you can skip this cell.
import sys, subprocess
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import ee, geopandas, numba, scipy, openpyxl, rasterio  # noqa
    print("Core packages already present.")
except ImportError:
    _pip("earthengine-api", "geopandas", "numba", "scipy", "openpyxl",
         "rasterio", "pyproj", "shapely", "matplotlib", "seaborn", "pandas")
    print("Installed core packages.")


## 2. Imports & runtime detection

In [ ]:
import os, json, math, io, zipfile, warnings, traceback
from pathlib import Path
from datetime import datetime, timedelta, date
from math import pi

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from numba import njit
from scipy.signal import savgol_filter
import geopandas as gpd
from shapely.geometry import Polygon, mapping
import ee

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")


## 3. Authenticate Google Earth Engine

You need a free Google Cloud project ID with the Earth Engine API enabled.
See https://developers.google.com/earth-engine/guides/access if this is
your first time.

The first call below will open a browser tab. Paste the verification code
back into the prompt.

In [ ]:
# >>> EDIT THIS LINE <<<
GCP_PROJECT_ID = "your-gcp-project-id"   # e.g. "my-wapor-class-2026"

try:
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine initialised.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine authenticated and initialised.")


## 4. Site & data-source configuration

Edit the values below. They control what data is pulled and how the
forecast is built.

In [ ]:
# ---- WaPOR / CHIRPS / ERA5 settings ------------------------------------
WAPOR_VERSION  = "v3"        # "v3" (recommended) or "v2"
WAPOR_LEVEL    = "L1"        # "L1" (300 m, global), "L2" (100 m, country/basin),
                             # or "L3" (30 m, selected AOIs only).
                             # RET is ALWAYS taken from L1 (only level where it exists).
PRECIP_SOURCE  = "CHIRPS"    # "CHIRPS" (daily, ~5 km) or "WAPOR_PCP" (dekadal)

# ---- Time window -------------------------------------------------------
HISTORICAL_LOOKBACK_YEARS = 5      # years of history used to build climatology
HISTORICAL_END_OFFSET     = 30     # days; WaPOR has ~30 day latency

# ---- Forecast ----------------------------------------------------------
MAX_FORECAST_DAYS = 365     # hard cap on forecast horizon
N_ENSEMBLE        = 20      # stochastic ensemble members for uncertainty fan

# ---- Irrigation system -------------------------------------------------
IRRIGATION_SYSTEM = 0        # 0 = Pressurized (drip/sprinkler), 1 = Surface (furrow/basin)

# Optional manual overrides (set to None to auto-derive from shapefile centroid):
MANUAL_LAT          = None
MANUAL_LON          = None
MANUAL_ELEVATION_M  = None

print("Configuration set.")


## 5. Upload your shapefiles

In Colab:

* **Easiest:** zip your soil shapefile (the .shp + .shx + .dbf + .prj all
  together) into `soil.zip`, the crop one into `crop.zip`, and run the
  `files.upload()` cell below.
* **Alternative:** open the file panel (folder icon on the left), create
  `/content/data/soil/` and `/content/data/crop/`, drag the four files of
  each shapefile in, and skip the upload cell.

In VS Code: place your shapefiles under `./data/soil/` and `./data/crop/`
relative to the notebook, and skip the upload cell.

In [ ]:
# Optional upload helper — only used in Colab.
DATA_ROOT = Path("/content/data") if IN_COLAB else Path("./data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "soil").mkdir(exist_ok=True)
(DATA_ROOT / "crop").mkdir(exist_ok=True)

def _unzip_to(zip_bytes: bytes, target: Path):
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        zf.extractall(target)

if IN_COLAB:
    from google.colab import files
    print("Choose soil.zip (cancel to skip):")
    try:
        up = files.upload()
        for name, data in up.items():
            if name.lower().endswith(".zip"):
                target = DATA_ROOT / ("soil" if "soil" in name.lower() else "crop")
                _unzip_to(data, target)
                print(f"Unzipped {name} -> {target}")
    except Exception as exc:
        print(f"Upload skipped: {exc}")

print("\nContents of data folder:")
for p in sorted(DATA_ROOT.rglob("*")):
    print("  ", p.relative_to(DATA_ROOT))


## 6. Read and validate the shapefiles

The notebook auto-detects any `.shp` under `data/soil/` and `data/crop/`,
checks the required attribute column, and reprojects to EPSG:4326 (lat/lon)
because the GEE queries work in geographic coordinates.

In [ ]:
def _find_shp(folder: Path) -> Path:
    shps = list(folder.glob("*.shp"))
    if not shps:
        raise FileNotFoundError(f"No .shp file found in {folder}")
    return shps[0]

def _read_zone_shapefile(folder: Path, required_col: str, label: str) -> gpd.GeoDataFrame:
    shp = _find_shp(folder)
    try:
        gdf = gpd.read_file(shp, engine="pyogrio")
    except Exception:
        gdf = gpd.read_file(shp)
    if required_col not in gdf.columns:
        # Be helpful: try common renames
        candidates = [c for c in gdf.columns if required_col.split('_')[0].lower() in c.lower()]
        raise ValueError(
            f"{label} shapefile is missing required column '{required_col}'. "
            f"Found columns: {list(gdf.columns)}. "
            f"Likely candidates to rename: {candidates}"
        )
    if gdf.crs is None:
        raise ValueError(f"{label} shapefile has no CRS. Please define one in GIS first.")
    if gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(4326)
        print(f"{label}: reprojected to EPSG:4326.")
    gdf[required_col] = gdf[required_col].astype(str)
    print(f"{label}: {len(gdf)} polygons, zones = {sorted(gdf[required_col].unique())}")
    return gdf

soil_gdf = _read_zone_shapefile(DATA_ROOT / "soil", "soil_zone", "Soil")
crop_gdf = _read_zone_shapefile(DATA_ROOT / "crop", "crop_zone", "Crop")

# Pre-compute the soil x crop intersection (each polygon = unique management unit)
intersection_gdf = gpd.overlay(soil_gdf, crop_gdf, how="intersection")
intersection_gdf["area_m2"] = intersection_gdf.to_crs(3857).geometry.area
print(f"\nIntersection polygons: {len(intersection_gdf)} unique soil x crop units")
intersection_gdf.head()


## 7. Define the area of interest and preview the geometry

In [ ]:
# Centroid of the union of all polygons -> default lat/lon for queries
try:
    soil_u = soil_gdf.union_all(); crop_u = crop_gdf.union_all()
except AttributeError:
    soil_u = soil_gdf.unary_union; crop_u = crop_gdf.unary_union
union_geom = soil_u.union(crop_u)
centroid = union_geom.centroid
LAT = MANUAL_LAT if MANUAL_LAT is not None else centroid.y
LON = MANUAL_LON if MANUAL_LON is not None else centroid.x

# Elevation: pull from SRTM at the centroid if not supplied
if MANUAL_ELEVATION_M is not None:
    ELEVATION_M = MANUAL_ELEVATION_M
else:
    point = ee.Geometry.Point([LON, LAT])
    srtm = ee.Image("USGS/SRTMGL1_003").select("elevation")
    ELEVATION_M = float(
        srtm.reduceRegion(ee.Reducer.first(), point, 30).get("elevation").getInfo()
    )

print(f"LAT = {LAT:.5f}    LON = {LON:.5f}    ELEVATION = {ELEVATION_M:.0f} m a.s.l.")

# AOI for GEE = bounding box of soil ∪ crop, slightly buffered
minx, miny, maxx, maxy = union_geom.bounds
buf = 0.01  # ~1 km
AOI = ee.Geometry.Rectangle([minx - buf, miny - buf, maxx + buf, maxy + buf])

# Static preview map
fig, ax = plt.subplots(1, 1, figsize=(8, 7))
soil_gdf.plot(ax=ax, facecolor="none", edgecolor="saddlebrown", linewidth=1.5, label="Soil")
crop_gdf.plot(ax=ax, facecolor="none", edgecolor="seagreen", linewidth=1.5, label="Crop")
intersection_gdf.plot(ax=ax, facecolor="khaki", edgecolor="black", alpha=0.4)
ax.scatter([LON], [LAT], color="red", marker="*", s=200, zorder=5, label="Centroid")
ax.set_title("Soil zones (brown), Crop zones (green), Intersections (yellow)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout(); plt.show()


## 8. WaPOR data architecture (what each level provides)

| Level | Resolution | Coverage         | RET     | AETI / E / T / I / NPP |
| ----- | ---------- | ---------------- | ------- | ----------------------- |
| L1    | 300 m      | Africa & Near-East | **daily** | dekadal (10-day)        |
| L2    | 100 m      | Selected basins/countries | _not available_ | dekadal |
| L3    | 30 m       | Selected farms/schemes | _not available_ | dekadal |

The notebook **always pulls RET from L1** (because higher levels do not
provide it) and pulls **AETI from your chosen level** for downscaled
validation. ERA5-Land supplies the daily Tmax / Tmin / wind that the
FAO-56 dual-Kc adjustment still needs (WaPOR does not expose these).

References:
* https://developers.google.com/earth-engine/datasets/catalog/FAO_WAPOR_3_L1_AETI_D
* https://developers.google.com/earth-engine/datasets/catalog/FAO_WAPOR_2_L1_AETI_D


## 9. Reference tables (soil properties and FAO-56 crop coefficients)

These mirror exactly what `app.py` ships with. You can edit any value.

In [ ]:
SOIL_PROPERTIES = {
    "Sand":            {"REW": 2, "TEW": 6},
    "Loamy sand":      {"REW": 4, "TEW": 9},
    "Sandy loam":      {"REW": 6, "TEW": 15},
    "Loam":            {"REW": 8, "TEW": 16},
    "Silt loam":       {"REW": 8, "TEW": 18},
    "Silt":            {"REW": 8, "TEW": 22},
    "Silt clay loam":  {"REW": 8, "TEW": 22},
    "Silty clay":      {"REW": 8, "TEW": 22},
    "Clay":            {"REW": 8, "TEW": 22},
}

# Inline copy of crop_data.json so the notebook is self-contained.
CROP_LIBRARY = {
    "Wheat":     {"kcb": {"initial":0.30,"mid":1.15,"end":0.40}, "height":{"initial":0.1,"mid":1.0,"end":0.5}, "lengths":{"initial":20,"development":55,"mid":65,"late":29}},
    "Maize":     {"kcb": {"initial":0.30,"mid":1.20,"end":0.60}, "height":{"initial":0.2,"mid":2.0,"end":1.5}, "lengths":{"initial":25,"development":40,"mid":45,"late":30}},
    "Soybean":   {"kcb": {"initial":0.15,"mid":1.10,"end":0.30}, "height":{"initial":0.1,"mid":0.65,"end":0.3}, "lengths":{"initial":15,"development":30,"mid":60,"late":25}},
    "Rice":      {"kcb": {"initial":0.15,"mid":1.20,"end":0.60}, "height":{"initial":0.1,"mid":1.0,"end":0.5}, "lengths":{"initial":30,"development":30,"mid":60,"late":30}},
    "Cotton":    {"kcb": {"initial":0.15,"mid":1.15,"end":0.50}, "height":{"initial":0.1,"mid":1.2,"end":1.0}, "lengths":{"initial":30,"development":50,"mid":60,"late":55}},
    "Tomato":    {"kcb": {"initial":0.15,"mid":1.15,"end":0.80}, "height":{"initial":0.1,"mid":0.6,"end":0.5}, "lengths":{"initial":30,"development":40,"mid":45,"late":30}},
    "Potato":    {"kcb": {"initial":0.15,"mid":1.10,"end":0.65}, "height":{"initial":0.1,"mid":0.6,"end":0.4}, "lengths":{"initial":25,"development":30,"mid":45,"late":30}},
    "Barley":    {"kcb": {"initial":0.30,"mid":1.15,"end":0.25}, "height":{"initial":0.1,"mid":1.0,"end":0.5}, "lengths":{"initial":15,"development":25,"mid":50,"late":30}},
    "Sorghum":   {"kcb": {"initial":0.15,"mid":1.10,"end":0.65}, "height":{"initial":0.1,"mid":1.5,"end":1.2}, "lengths":{"initial":20,"development":35,"mid":40,"late":30}},
    "Sunflower": {"kcb": {"initial":0.15,"mid":1.15,"end":0.35}, "height":{"initial":0.1,"mid":2.0,"end":1.5}, "lengths":{"initial":25,"development":35,"mid":45,"late":25}},
}
print(f"Crops in library: {list(CROP_LIBRARY)}")
print(f"Soil texture classes: {list(SOIL_PROPERTIES)}")


## 10. Configure each zone

For every `soil_zone` in `soil_gdf` and every `crop_zone` in `crop_gdf`,
define one entry in the dictionaries below. The defaults pre-populate with
plausible values; **edit them**.

`planting_date` may be in the past, today, or in the future. The total
season length = sum of the four `lengths` values, and the derived
**harvest date = planting_date + total length**.

Set `harvest_date_override` to a string like `"2026-09-15"` to truncate
the season earlier (otherwise leave as `None`).

In [ ]:
# >>> EDIT THIS DICTIONARY — one entry per soil_zone in your shapefile <<<
soil_configs = {}
for z in sorted(soil_gdf["soil_zone"].unique()):
    soil_configs[z] = {
        "soil_type": "Loam",          # any key from SOIL_PROPERTIES
        "TAW":       120.0,           # mm; total available water in root zone
        "p":         0.55,            # depletion fraction
    }
# Auto-fill REW/TEW from the texture choice
for z, cfg in soil_configs.items():
    cfg["REW"] = SOIL_PROPERTIES[cfg["soil_type"]]["REW"]
    cfg["TEW"] = SOIL_PROPERTIES[cfg["soil_type"]]["TEW"]

# >>> EDIT THIS DICTIONARY — one entry per crop_zone in your shapefile <<<
crop_configs = {}
for z in sorted(crop_gdf["crop_zone"].unique()):
    base = CROP_LIBRARY["Maize"]      # change the crop name here
    crop_configs[z] = {
        "name":          "Maize",
        "planting_date": "2025-04-01",        # YYYY-MM-DD; past/today/future all OK
        "harvest_date_override": None,        # or "YYYY-MM-DD"
        "lengths":       dict(base["lengths"]),
        "kcb":           dict(base["kcb"]),
        "height":        dict(base["height"]),
    }

# Quick summary
print("Soil configs:")
for z, cfg in soil_configs.items():
    print(f"  zone {z}: {cfg}")
print("Crop configs:")
for z, cfg in crop_configs.items():
    total = sum(cfg["lengths"].values())
    plant = pd.to_datetime(cfg["planting_date"]).date()
    harvest = plant + timedelta(days=total)
    print(f"  zone {z}: {cfg['name']}  planting={plant}  total_season={total}d  harvest={harvest}")


## 11. Derive the time window automatically from the crop calendar

The notebook needs weather covering the **earliest planting date** to the
**latest harvest date** across all crops. It works out the split between
*observed* (history) and *forecast* (future) automatically.

In [ ]:
today = date.today()

planting_dates = [pd.to_datetime(c["planting_date"]).date() for c in crop_configs.values()]
season_lengths = [sum(c["lengths"].values()) for c in crop_configs.values()]
harvest_dates  = []
for c in crop_configs.values():
    plant   = pd.to_datetime(c["planting_date"]).date()
    total   = sum(c["lengths"].values())
    harvest = plant + timedelta(days=total)
    if c["harvest_date_override"]:
        harvest = min(harvest, pd.to_datetime(c["harvest_date_override"]).date())
    harvest_dates.append(harvest)

SEASON_START = min(planting_dates)
SEASON_END   = max(harvest_dates)

# What WaPOR can give us as observation (latency ~30 days)
HIST_END = today - timedelta(days=HISTORICAL_END_OFFSET)
HIST_START_FOR_CLIMATOLOGY = today - timedelta(days=365 * HISTORICAL_LOOKBACK_YEARS)

# We need observations from min(SEASON_START, climatology start) up to HIST_END
OBS_START = min(SEASON_START, HIST_START_FOR_CLIMATOLOGY)
OBS_END   = min(SEASON_END,   HIST_END)

# Forecast covers anything from HIST_END+1 up to SEASON_END (capped)
FORECAST_START = OBS_END + timedelta(days=1)
FORECAST_END   = min(SEASON_END, today + timedelta(days=MAX_FORECAST_DAYS))

print(f"Today                     : {today}")
print(f"Earliest planting date    : {SEASON_START}")
print(f"Latest harvest date       : {SEASON_END}")
print(f"Observation window        : {OBS_START}  ->  {OBS_END}")
if FORECAST_END >= FORECAST_START:
    fc_days = (FORECAST_END - FORECAST_START).days + 1
    print(f"Forecast window           : {FORECAST_START}  ->  {FORECAST_END}  ({fc_days} days)")
else:
    print("Forecast window           : none (entire season is in the past)")


## 12. GEE helpers — pull WaPOR + CHIRPS + ERA5-Land into a daily DataFrame

Each helper returns a `pandas.Series` indexed by date. They are pure
functions; rerun any of them independently to inspect a single variable.

In [ ]:
def _ic_to_series(ic, band, scale, aoi=AOI, label=""):
    """Reduce an ImageCollection to a (date, mean) pandas Series over the AOI."""
    def _per_image(img):
        val = img.select(band).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=aoi, scale=scale, maxPixels=1e10
        ).get(band)
        return ee.Feature(None, {"date": img.date().format("YYYY-MM-dd"), "v": val})
    feats = ic.map(_per_image).filter(ee.Filter.notNull(["v"])).getInfo()["features"]
    if not feats:
        return pd.Series(dtype=float, name=label)
    s = pd.Series(
        {f["properties"]["date"]: f["properties"]["v"] for f in feats},
        name=label,
    )
    s.index = pd.to_datetime(s.index)
    return s.sort_index()

def fetch_wapor_ret_daily(start, end):
    """WaPOR L1 RET (daily reference ET, mm/day). v3 or v2 chosen by config."""
    asset = "FAO/WAPOR/3/L1/RET_D" if WAPOR_VERSION == "v3" else "FAO/WAPOR/2/L1/RET_D"
    ic = ee.ImageCollection(asset).filterDate(str(start), str(end + timedelta(days=1)))
    band = "RET" if WAPOR_VERSION == "v3" else "L1_RET_D"
    s = _ic_to_series(ic, band, scale=300, label="ETo")
    # WaPOR scale factor: v3 RET is in 0.01 mm/day, v2 RET in 0.1 mm/day
    scale_factor = 0.01 if WAPOR_VERSION == "v3" else 0.1
    return s * scale_factor

def fetch_wapor_aeti(start, end):
    """WaPOR AETI (dekadal actual ET, mm/dekad). Uses chosen LEVEL."""
    if WAPOR_VERSION == "v3":
        asset = f"FAO/WAPOR/3/{WAPOR_LEVEL}/AETI_D"
        band  = "AETI"
        scale = {"L1": 300, "L2": 100, "L3": 30}[WAPOR_LEVEL]
        scale_factor = 0.1
    else:
        asset = f"FAO/WAPOR/2/{WAPOR_LEVEL}/AETI_D"
        band  = f"{WAPOR_LEVEL}_AETI_D"
        scale = {"L1": 250, "L2": 100, "L3": 30}[WAPOR_LEVEL]
        scale_factor = 0.1
    try:
        ic = ee.ImageCollection(asset).filterDate(str(start), str(end + timedelta(days=1)))
        s = _ic_to_series(ic, band, scale=scale, label="AETI_dekad")
        return s * scale_factor
    except Exception as exc:
        print(f"AETI not available at {WAPOR_LEVEL} for this AOI ({exc}). "
              f"Falling back to L1.")
        ic = ee.ImageCollection(
            "FAO/WAPOR/3/L1/AETI_D" if WAPOR_VERSION == "v3" else "FAO/WAPOR/2/L1/AETI_D"
        ).filterDate(str(start), str(end + timedelta(days=1)))
        return _ic_to_series(ic, "AETI" if WAPOR_VERSION == "v3" else "L1_AETI_D",
                             scale=300, label="AETI_dekad") * 0.1

def fetch_chirps_daily(start, end):
    ic = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
          .filterDate(str(start), str(end + timedelta(days=1))))
    return _ic_to_series(ic, "precipitation", scale=5500, label="P")

def fetch_wapor_pcp(start, end):
    asset = "FAO/WAPOR/3/L1/PCP_E" if WAPOR_VERSION == "v3" else "FAO/WAPOR/2/L1/PCP_E"
    band  = "PCP"                if WAPOR_VERSION == "v3" else "L1_PCP_E"
    ic = ee.ImageCollection(asset).filterDate(str(start), str(end + timedelta(days=1)))
    s = _ic_to_series(ic, band, scale=300, label="P_dekad")
    return s * 0.1

def fetch_era5_daily(start, end):
    """Tmax, Tmin (°C), and 10 m wind speed converted to 2 m wind (m/s)."""
    ic = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
          .filterDate(str(start), str(end + timedelta(days=1))))
    tmax = _ic_to_series(ic, "temperature_2m_max", scale=10000, label="Tmax_K") - 273.15
    tmin = _ic_to_series(ic, "temperature_2m_min", scale=10000, label="Tmin_K") - 273.15
    u10  = _ic_to_series(ic, "u_component_of_wind_10m", scale=10000, label="u10")
    v10  = _ic_to_series(ic, "v_component_of_wind_10m", scale=10000, label="v10")
    ws10 = np.sqrt(u10**2 + v10**2)
    # Allen et al. 1998 eq. 47: u2 = u10 * 4.87 / ln(67.8*z - 5.42), z=10 m
    u2 = ws10 * 4.87 / np.log(67.8 * 10 - 5.42)
    return tmax.rename("Tmax"), tmin.rename("Tmin"), u2.rename("u2")

print("GEE helpers defined.")


## 13. Pull the observed (historical) data

This single cell does all the GEE traffic. On a typical AOI (a few km²)
and a 5-year history it takes 30–90 s. If GEE rate-limits, just rerun.

In [ ]:
print(f"Pulling RET daily   ({OBS_START} -> {OBS_END}) ...")
eto_s   = fetch_wapor_ret_daily(OBS_START, OBS_END)
print(f"  -> {len(eto_s)} observations")

print(f"Pulling AETI dekadal ({WAPOR_LEVEL}) ...")
aeti_s  = fetch_wapor_aeti(OBS_START, OBS_END)
print(f"  -> {len(aeti_s)} dekads")

if PRECIP_SOURCE == "CHIRPS":
    print("Pulling CHIRPS daily precipitation ...")
    p_s = fetch_chirps_daily(OBS_START, OBS_END)
else:
    print("Pulling WaPOR PCP dekadal precipitation ...")
    p_dekad = fetch_wapor_pcp(OBS_START, OBS_END)
    # disaggregate dekad totals evenly to daily for SWB
    daily_idx = pd.date_range(OBS_START, OBS_END, freq="D")
    p_s = p_dekad.reindex(daily_idx, method="bfill") / 10.0
    p_s.name = "P"
print(f"  -> {len(p_s)} daily precipitation values, total {p_s.sum():.0f} mm")

print("Pulling ERA5-Land Tmax / Tmin / u2 ...")
tmax_s, tmin_s, u2_s = fetch_era5_daily(OBS_START, OBS_END)
print(f"  -> {len(tmax_s)} daily T records")

# Assemble a single daily DataFrame, indexed on a continuous date range.
daily_idx = pd.date_range(OBS_START, OBS_END, freq="D")
hist_df = pd.DataFrame(index=daily_idx)
hist_df["ETo"]  = eto_s.reindex(daily_idx).interpolate("time")
hist_df["P"]    = p_s.reindex(daily_idx).fillna(0.0)
hist_df["Tmax"] = tmax_s.reindex(daily_idx).interpolate("time")
hist_df["Tmin"] = tmin_s.reindex(daily_idx).interpolate("time")
hist_df["u2"]   = u2_s.reindex(daily_idx).interpolate("time")

# Solar radiation Rs estimated via Hargreaves (kept consistent with app.py)
hist_df["Rs"] = 0.16 * np.sqrt((hist_df["Tmax"] - hist_df["Tmin"]).clip(lower=0)) * 20.0
# Vapour pressure from Tmin (FAO-56 default proxy)
hist_df["Vap_P"] = 0.6108 * np.exp(17.27 * hist_df["Tmin"] / (hist_df["Tmin"] + 237.3))

hist_df = hist_df.reset_index().rename(columns={"index": "Date"})
print("\nObserved daily DataFrame ready:")
print(hist_df.head())
print(f"... {len(hist_df)} days from {hist_df['Date'].min().date()} to {hist_df['Date'].max().date()}")

# Quick sanity plot
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(hist_df["Date"], hist_df["ETo"], color="darkorange", lw=0.8)
axes[0].set_ylabel("ETo (mm/d)")
axes[0].set_title("WaPOR daily reference ET")
axes[1].bar(hist_df["Date"], hist_df["P"], color="steelblue", width=1.0)
axes[1].set_ylabel("P (mm/d)"); axes[1].set_title(f"{PRECIP_SOURCE} daily precipitation")
plt.tight_layout(); plt.show()


## 14. Build a climatological forecast

Method:

1. Index every observed day by its day-of-year `doy = 1..366`.
2. For each `doy`, compute a 7-day-window mean and standard deviation of
   `ETo, Tmax, Tmin, u2`, and a probability-of-rain + mean rain-on-rainy-day
   for `P` (a simple two-state stochastic model).
3. The deterministic forecast = climatology mean.
4. Optional ensemble: `N_ENSEMBLE` stochastic members sampled from the
   distributions, used to draw an uncertainty fan.

In [ ]:
def build_climatology(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["doy"] = df["Date"].dt.dayofyear
    g = df.groupby("doy")
    clim = pd.DataFrame({
        "ETo_mean":  g["ETo"].mean(),
        "ETo_std":   g["ETo"].std().fillna(0.0),
        "Tmax_mean": g["Tmax"].mean(),
        "Tmax_std":  g["Tmax"].std().fillna(0.0),
        "Tmin_mean": g["Tmin"].mean(),
        "Tmin_std":  g["Tmin"].std().fillna(0.0),
        "u2_mean":   g["u2"].mean(),
        "u2_std":    g["u2"].std().fillna(0.0),
        "P_prob":    g["P"].apply(lambda s: float((s > 0.1).mean())),
        "P_mean_wet":g["P"].apply(lambda s: float(s[s > 0.1].mean()) if (s > 0.1).any() else 0.0),
    })
    # Smooth the climatology with a 7-day centred rolling mean
    clim = clim.rolling(7, center=True, min_periods=1).mean()
    return clim

def sample_forecast(clim: pd.DataFrame, start: date, end: date,
                    rng: np.random.Generator | None = None,
                    deterministic: bool = True) -> pd.DataFrame:
    if rng is None:
        rng = np.random.default_rng(42)
    dates = pd.date_range(start, end, freq="D")
    rows = []
    for d in dates:
        doy = d.dayofyear
        c = clim.loc[doy] if doy in clim.index else clim.iloc[(np.abs(clim.index - doy)).argmin()]
        if deterministic:
            eto = c["ETo_mean"]
            tmax = c["Tmax_mean"]; tmin = c["Tmin_mean"]
            u2  = c["u2_mean"]
            p   = c["P_prob"] * c["P_mean_wet"]
        else:
            eto = max(0.0, rng.normal(c["ETo_mean"], c["ETo_std"]))
            tmax = rng.normal(c["Tmax_mean"], c["Tmax_std"])
            tmin = rng.normal(c["Tmin_mean"], c["Tmin_std"])
            u2  = max(0.1, rng.normal(c["u2_mean"], c["u2_std"]))
            wet = rng.random() < c["P_prob"]
            p   = max(0.0, rng.gamma(2.0, c["P_mean_wet"]/2.0)) if wet and c["P_mean_wet"] > 0 else 0.0
        tmin = min(tmin, tmax - 1.0)  # enforce Tmin < Tmax
        rs   = 0.16 * np.sqrt(max(tmax - tmin, 0.0)) * 20.0
        vap  = 0.6108 * np.exp(17.27 * tmin / (tmin + 237.3))
        rows.append([d, tmax, tmin, rs, u2, p, vap, eto])
    return pd.DataFrame(rows, columns=["Date","Tmax","Tmin","Rs","u2","P","Vap_P","ETo"])

clim = build_climatology(hist_df)
print(f"Climatology built from {len(hist_df)} observed days.")

if FORECAST_END >= FORECAST_START:
    fc_df = sample_forecast(clim, FORECAST_START, FORECAST_END, deterministic=True)
    # Ensemble for the uncertainty fan
    rng = np.random.default_rng(2026)
    ens = [sample_forecast(clim, FORECAST_START, FORECAST_END, rng=rng, deterministic=False)
           for _ in range(N_ENSEMBLE)]

    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
    for member in ens:
        axes[0].plot(member["Date"], member["ETo"], color="orange", alpha=0.15)
        axes[1].plot(member["Date"], member["P"].cumsum(), color="steelblue", alpha=0.15)
    axes[0].plot(fc_df["Date"], fc_df["ETo"], color="red", lw=1.5, label="Deterministic")
    axes[0].set_ylabel("ETo (mm/d)"); axes[0].set_title("Forecast: ETo (ensemble fan)")
    axes[0].legend()
    axes[1].plot(fc_df["Date"], fc_df["P"].cumsum(), color="navy", lw=1.5, label="Deterministic")
    axes[1].set_ylabel("Cumulative P (mm)"); axes[1].set_title("Forecast: cumulative precipitation")
    axes[1].legend()
    plt.tight_layout(); plt.show()
else:
    fc_df = pd.DataFrame(columns=hist_df.columns)
    print("No forecast needed.")


## 15. Final weather DataFrame `weather_df` (history + forecast)

In [ ]:
cols = ["Date","Tmax","Tmin","Rs","u2","P","Vap_P","ETo"]
weather_df = pd.concat([hist_df[cols], fc_df[cols]], ignore_index=True)
weather_df = weather_df.sort_values("Date").reset_index(drop=True)

# Mark which rows are observed vs forecast (handy for plotting)
cutoff = OBS_END
weather_df["source"] = np.where(weather_df["Date"].dt.date <= cutoff, "observed", "forecast")

print(f"weather_df: {len(weather_df)} rows  "
      f"({(weather_df.source=='observed').sum()} observed, "
      f"{(weather_df.source=='forecast').sum()} forecast)")
print(f"Range: {weather_df['Date'].min().date()}  ->  {weather_df['Date'].max().date()}")
weather_df.head()


## 16. FAO-56 engine — part A: ETo (Penman–Monteith), Kcb, fc

Identical equations to `app.py`, refactored to take `lat` and `h` as
explicit arguments (no `st.session_state`). You only need to run this
cell once per session.

The Penman–Monteith equation implemented here is FAO Irrigation & Drainage
Paper 56, Eq. 6:

$$
ETo = \frac{0.408\,\Delta\,(R_n - G) + \gamma\,\frac{900}{T+273}\,u_2\,(e_s - e_a)}
            {\Delta + \gamma\,(1 + 0.34\,u_2)}
$$


In [ ]:
def calculate_PMETo(df: pd.DataFrame, lat: float, h: float) -> pd.Series:
    """FAO-56 Penman-Monteith reference evapotranspiration (mm/day)."""
    lat_rad = lat * pi / 180.0
    P = 101.3 * ((293 - 0.0065 * h) / 293) ** 5.26
    gamma = 0.000665 * P
    Tav = (df["Tmax"] + df["Tmin"]) / 2.0
    es  = (0.6108 * np.exp(17.27 * df["Tmax"] / (df["Tmax"] + 237.3)) +
           0.6108 * np.exp(17.27 * df["Tmin"] / (df["Tmin"] + 237.3))) / 2.0
    delta = 4098 * 0.6108 * np.exp(17.27 * Tav / (Tav + 237.3)) / (Tav + 237.3) ** 2
    Ra = df["Ra"]
    Rn = ((1 - 0.23) * df["Rs"] -
          ((4.903e-9) * ((df["Tmax"] + 273.16) ** 4 + (df["Tmin"] + 273.16) ** 4) / 2) *
          (0.34 - 0.14 * np.sqrt(df["Vap_P"])) *
          (1.35 * df["Rs"] / ((0.25 + 0.54) * Ra) - 0.35))
    eto = ((0.408 * delta * Rn + gamma * 900 / (Tav + 273) * df["u2"] * (es - df["Vap_P"])) /
           (delta + gamma * (1 + 0.34 * df["u2"])))
    return eto

def calculate_kcb(df, initial_length, dev_length, mid_length, late_length,
                  initial_value, dev_value, mid_value, late_value):
    df["kcb"] = np.nan
    total = initial_length + dev_length + mid_length + late_length
    mask = df["DAP"] <= total
    season = df[mask].copy()
    if season.empty:
        return df
    initial_end = initial_length
    dev_end = initial_end + dev_length
    mid_end = dev_end + mid_length
    late_end = mid_end + late_length
    x_pts = [1, initial_end, initial_end+5, dev_end-5, dev_end, mid_end-5, mid_end, late_end]
    y_pts = [initial_value, initial_value, initial_value, mid_value,
             mid_value, mid_value, mid_value, late_value]
    season["kcb"] = np.interp(season["DAP"].values, x_pts, y_pts)
    season["kcb"] = season["kcb"].rolling(5, center=True, min_periods=1).mean()
    season["kcb"] = season["kcb"].clip(lower=0.0, upper=max(mid_value, initial_value, late_value))
    df.loc[mask, "kcb"] = season["kcb"].values
    return df

def calculate_h_crop(df, initial_length, dev_length, mid_length, late_length,
                     initial_value, dev_value, mid_value, late_value):
    df["h_crop"] = 0.3
    initial_end = initial_length
    dev_end = initial_end + dev_length
    mid_end = dev_end + mid_length
    late_end = mid_end + late_length
    df.loc[df["DAP"] <= initial_end, "h_crop"] = initial_value
    df.loc[(df["DAP"] > initial_end) & (df["DAP"] <= dev_end),  "h_crop"] = dev_value
    df.loc[(df["DAP"] > dev_end)     & (df["DAP"] <= mid_end),  "h_crop"] = mid_value
    df.loc[(df["DAP"] > mid_end)     & (df["DAP"] <= late_end), "h_crop"] = late_value
    return df

def process_weather_data(df: pd.DataFrame, crop_props: dict | None,
                         lat: float, h: float, use_wapor_eto: bool = True) -> pd.DataFrame:
    """Add Ra, Vap_P, ETo (or override with WaPOR), DAP, Kcb, kc_max, fc, ETc."""
    needed = ["Date","Tmax","Tmin","Rs","u2","P"]
    df_slim = df[needed + (["ETo"] if "ETo" in df.columns else [])].copy()
    df_slim["Date"] = pd.to_datetime(df_slim["Date"])
    df_slim = df_slim.dropna(subset=needed).reset_index(drop=True)

    df_slim["day_of_year"] = df_slim["Date"].dt.dayofyear
    lat_rad = lat * pi / 180.0
    df_slim["D"]  = 0.409 * np.sin(2*pi*df_slim["day_of_year"]/365 - 1.39)
    df_slim["H"]  = np.arccos(-np.tan(lat_rad) * np.tan(df_slim["D"]))
    df_slim["N"]  = 24 * df_slim["H"] / pi
    df_slim["Ra"] = (24*60/pi) * 0.082 * (
        df_slim["H"]*np.sin(lat_rad)*np.sin(df_slim["D"]) +
        np.cos(lat_rad)*np.cos(df_slim["D"])*np.sin(df_slim["H"])
    )
    df_slim["Vap_P"] = 0.6108 * np.exp(17.27 * df_slim["Tmin"] / (df_slim["Tmin"] + 237.3))

    pm_eto = calculate_PMETo(df_slim, lat, h)
    if use_wapor_eto and "ETo" in df_slim.columns and df_slim["ETo"].notna().any():
        # Prefer WaPOR RET where present; fall back to PM-ETo for any NaN
        df_slim["ETo"] = df_slim["ETo"].fillna(pm_eto)
    else:
        df_slim["ETo"] = pm_eto

    if crop_props is None:
        return df_slim

    planting = pd.to_datetime(crop_props["planting_date"])
    df_slim["DAP"] = (df_slim["Date"] - planting).dt.days + 1
    df_slim = df_slim[df_slim["DAP"] > 0].copy()
    total = sum(crop_props["lengths"].values())
    df_slim = df_slim[df_slim["DAP"] <= total].copy()

    df_slim = calculate_h_crop(df_slim,
        crop_props["lengths"]["initial"], crop_props["lengths"]["development"],
        crop_props["lengths"]["mid"],     crop_props["lengths"]["late"],
        crop_props["height"]["initial"],  crop_props["height"]["mid"],
        crop_props["height"]["mid"],      crop_props["height"]["end"])

    df_slim = calculate_kcb(df_slim,
        crop_props["lengths"]["initial"], crop_props["lengths"]["development"],
        crop_props["lengths"]["mid"],     crop_props["lengths"]["late"],
        crop_props["kcb"]["initial"],     crop_props["kcb"]["mid"],
        crop_props["kcb"]["mid"],         crop_props["kcb"]["end"])

    # Smoothed climate-corrected coefficients (FAO-56 dual-Kc Eq. 70)
    win = 5
    df_slim["ETo_smoothed"] = df_slim["ETo"].rolling(win, center=True, min_periods=1).mean()
    df_slim["kcb_smoothed"] = df_slim["kcb"].rolling(win, center=True, min_periods=1).mean()
    df_slim["u2_smoothed"]  = df_slim["u2"].ewm(alpha=0.3, adjust=False).mean()
    Rhmin = (df_slim["Vap_P"] / (0.6108 * np.exp(17.27 * df_slim["Tmax"] / (df_slim["Tmax"] + 237.3))) * 100)
    df_slim["Rhmin_smoothed"] = Rhmin.ewm(alpha=0.3, adjust=False).mean()
    df_slim["kcb_corr"] = (df_slim["kcb_smoothed"] +
        (0.04*(df_slim["u2_smoothed"]-2) - 0.004*(df_slim["Rhmin_smoothed"]-45)) *
        (df_slim["h_crop"]/3) ** 0.3).clip(lower=0.0, upper=1.4)
    df_slim["kc_max"] = np.maximum(
        1.2 + (0.04*(df_slim["u2_smoothed"]-2) - 0.004*(df_slim["Rhmin_smoothed"]-45)) *
        (df_slim["h_crop"]/3) ** 0.3,
        df_slim["kcb_corr"] + 0.05)
    df_slim["fc"] = (((df_slim["kcb_corr"]-0.15) / (df_slim["kc_max"]-0.15)) **
                     (1 + 0.5*df_slim["h_crop"])).clip(lower=0.0, upper=1.0)
    df_slim["ETc"] = df_slim["kcb_corr"] * df_slim["ETo_smoothed"]
    if len(df_slim) >= 7:
        df_slim["ETc"] = savgol_filter(df_slim["ETc"].values, 7, 3)
    df_slim["ETc"] = df_slim["ETc"].clip(lower=0.0).interpolate("linear", limit_direction="both")
    return df_slim

print("Engine A defined: calculate_PMETo, calculate_kcb, calculate_h_crop, process_weather_data.")


## 17. FAO-56 engine — part B: dual-Kc soil water balance

This is the JIT-compiled core (`calculate_dri_jit`) that produces the
recommended irrigation events. Two system branches:

* `system_type = 0` — pressurized (drip / sprinkler): multi-condition
  trigger, irrigation capped at 25 mm.
* `system_type = 1` — surface (furrow / basin): strict trigger, full refill
  to field capacity (no cap).

In [ ]:
@njit(cache=True)
def calculate_dri_jit(eto, p, kc_max, kcb_corr, fc, rew, tew, p_val, taw,
                      taw_max, taw_min, trigger_values, trigger_dap_values,
                      n, system_type):
    de   = np.zeros(n); dpe = np.zeros(n); kr = np.zeros(n); ke = np.zeros(n)
    e    = np.zeros(n); dei = np.zeros(n); kc = np.zeros(n); dr = np.zeros(n)
    ks   = np.zeros(n); etc = np.zeros(n); dp = np.zeros(n); irr = np.zeros(n)
    dri  = np.zeros(n)
    prev_de = 0.0; prev_dr = 0.0; days_since_irr = 0
    for i in range(n):
        dpe[i] = max(0.0, p[i] - prev_de)
        kr[i]  = 1.0 if prev_de <= rew else max(0.0, min(1.0, (tew - prev_de) / (tew - rew)))
        ke_val = kr[i] * (kc_max[i] - kcb_corr[i])
        fc_val = (1.0 - fc[i]) * kc_max[i]
        ke[i]  = min(ke_val, fc_val)
        e[i]   = eto[i] * ke[i]
        etc[i] = (kcb_corr[i] + ke[i]) * eto[i]
        # locate trigger band
        trigger_pos = 0
        for j in range(len(trigger_dap_values) - 1):
            if (i + 1) >= trigger_dap_values[j] and (i + 1) < trigger_dap_values[j + 1]:
                trigger_pos = j; break
        trigger_val = trigger_values[trigger_pos]

        should_irrigate = False; irr_amount = 0.0
        if system_type == 0:
            if prev_dr > trigger_val:
                should_irrigate = True
            elif days_since_irr >= 7 and prev_dr > 0.5 * trigger_val:
                should_irrigate = True
            elif eto[i] > 8.0 and prev_dr > 0.4 * trigger_val:
                should_irrigate = True
            if should_irrigate:
                target_deficit = 0.2 * taw
                irr_amount = min(25.0, prev_dr - target_deficit)
                irr_amount = max(10.0, irr_amount)
                days_since_irr = 0
            else:
                days_since_irr += 1
        else:
            if prev_dr > trigger_val:
                should_irrigate = True
            if should_irrigate:
                irr_amount = max(20.0, prev_dr)
                days_since_irr = 0
            else:
                days_since_irr += 1
        irr[i] = irr_amount
        dp[i]  = max(0.0, p[i] + irr[i] - etc[i] - (taw - prev_dr))
        dri[i] = prev_dr - p[i] - irr[i] + etc[i] + dp[i]
        dri[i] = max(0.0, min(dri[i], taw))
        prev_dr = dri[i]; dr[i] = prev_dr
        dei[i]  = min(tew, prev_de - p[i] + dpe[i] + e[i])
        prev_de = dei[i]
        kc[i]   = kcb_corr[i] + ke[i]
    return de, dpe, kr, ke, e, dei, kc, dr, ks, etc, dp, irr, dri


def calculate_Dri_optimized(ETo, P, kc_max, kcb_corr, fc, REW, TEW, p, TAW,
                            TAW_max, TAW_min, trigger_values, trigger_dap_values,
                            soil_zone, system_type=0):
    ETo_np      = np.asarray(ETo, dtype=np.float64).ravel()
    P_np        = np.asarray(P, dtype=np.float64).ravel()
    kc_max_np   = np.asarray(kc_max, dtype=np.float64).ravel()
    kcb_corr_np = np.asarray(kcb_corr, dtype=np.float64).ravel()
    fc_np       = np.asarray(fc, dtype=np.float64).ravel()
    n = min(len(ETo_np), len(P_np), len(kc_max_np), len(kcb_corr_np), len(fc_np))
    base = np.array([0.30*TAW, 0.35*TAW, 0.40*TAW, 0.50*TAW])
    adjusted = np.minimum(base, 0.75 * TAW)
    De, DPe, kr, ke, E, Dei, kc, Dr, ks, ETc, DP, IRR, Dri = calculate_dri_jit(
        ETo_np[:n], P_np[:n], kc_max_np[:n], kcb_corr_np[:n], fc_np[:n],
        REW, TEW, p, TAW, TAW_max, TAW_min,
        adjusted, trigger_dap_values, n, system_type)
    return pd.DataFrame({
        "soil_zone": [str(soil_zone)] * n,
        "De": De, "DPe": DPe, "kr": kr, "ke": ke, "E": E, "Dei": Dei,
        "kc": kc, "Dr": Dr, "ks": ks, "ETc": ETc, "DP": DP, "IRR": IRR, "Dri": Dri,
        "trigger_level": np.interp(np.arange(n), trigger_dap_values, adjusted),
    })


def calculate_zone_irrigation(weather_df, zone_data, soil_props, crop_props,
                              lat, h, system_type=0):
    df_temp = process_weather_data(weather_df.copy(), crop_props, lat, h)
    if df_temp is None or df_temp.empty:
        raise ValueError("process_weather_data returned no rows for this zone")
    m = len(df_temp)
    REW = soil_props["REW"]; TEW = soil_props["TEW"]
    p   = soil_props["p"];   TAW = soil_props["TAW"]
    initial_stage = crop_props["lengths"]["initial"]
    dev_stage     = crop_props["lengths"]["development"]
    mid_stage     = crop_props["lengths"]["mid"]
    trigger_dap_values = np.array([1, initial_stage,
                                   initial_stage + dev_stage,
                                   initial_stage + dev_stage + mid_stage])
    if system_type == 0:
        trigger_pct = np.array([0.11, 0.30, 0.30, 0.40])
    else:
        trigger_pct = np.array([0.40, 0.50, 0.50, 0.60])
    trigger_values = TAW * trigger_pct
    res = calculate_Dri_optimized(
        df_temp["ETo"].values[:m], df_temp["P"].values[:m],
        df_temp["kc_max"].values[:m], df_temp["kcb_corr"].values[:m],
        df_temp["fc"].values[:m], REW, TEW, p, TAW, TAW, TAW,
        trigger_values, trigger_dap_values,
        zone_data["soil_zone"], system_type)
    res["TAW"] = TAW; res["REW"] = REW; res["TEW"] = TEW; res["p"] = p
    res["crop_type"]    = zone_data["crop_type"]
    res["Date"]         = df_temp["Date"].values[:m]
    res["DAP"]          = df_temp["DAP"].values[:m]
    res["P"]            = df_temp["P"].values[:m]
    res["ETo_used"]     = df_temp["ETo"].values[:m]
    return res


def calculate_spatiotemporal_optimized(weather_df, soil_gdf, crop_gdf,
                                       soil_configs, crop_configs,
                                       lat, h, system_type=0):
    intersection = gpd.overlay(soil_gdf, crop_gdf, how="intersection")
    intersection["area"] = intersection.to_crs(3857).geometry.area
    out = []
    for crop_zone, crop_props in crop_configs.items():
        sub = intersection[intersection["crop_zone"] == crop_zone]
        for _, row in sub.iterrows():
            soil_zone = row["soil_zone"]
            soil_props = soil_configs.get(soil_zone)
            if soil_props is None:
                continue
            zone_data = {
                "soil_zone":     soil_zone,
                "crop_zone":     crop_zone,
                "crop_type":     crop_props.get("name", f"Crop {crop_zone}"),
                "area":          row["area"],
                "planting_date": crop_props["planting_date"],
            }
            res = calculate_zone_irrigation(weather_df, zone_data,
                                            soil_props, crop_props,
                                            lat, h, system_type)
            res["soil_zone"]     = soil_zone
            res["crop_zone"]     = crop_zone
            res["crop_type"]     = zone_data["crop_type"]
            res["area"]          = zone_data["area"]
            res["planting_date"] = crop_props["planting_date"]
            out.append(res)
    if not out:
        raise ValueError("No valid soil x crop combinations.")
    combined = pd.concat(out, ignore_index=True)
    combined["Date"] = pd.to_datetime(combined["Date"])
    return combined.sort_values(["crop_zone","soil_zone","Date"]).reset_index(drop=True)

print("Engine B defined: calculate_dri_jit, calculate_Dri_optimized, "
      "calculate_zone_irrigation, calculate_spatiotemporal_optimized.")


## 18. Per-crop preview — Kcb, kc_max, fc, ETc curves

Inspect the crop curve for each `crop_zone` *before* running the full SWB.

In [ ]:
for cz, cprops in crop_configs.items():
    df_prev = process_weather_data(weather_df.copy(), cprops, LAT, ELEVATION_M)
    if df_prev.empty:
        print(f"Crop zone {cz}: no data inside the season window. Check planting_date.")
        continue
    fig, ax1 = plt.subplots(figsize=(10, 3.5))
    ax2 = ax1.twinx()
    ax1.plot(df_prev["DAP"], df_prev["kcb_corr"], color="forestgreen", label="Kcb_corr")
    ax1.plot(df_prev["DAP"], df_prev["kc_max"],   color="darkred",     label="Kc_max")
    ax1.plot(df_prev["DAP"], df_prev["fc"],       color="goldenrod",   label="fc")
    ax2.plot(df_prev["DAP"], df_prev["ETc"],      color="steelblue", lw=1.5, label="ETc (mm/d)")
    ax1.set_xlabel("Days after planting"); ax1.set_ylabel("Coefficient")
    ax2.set_ylabel("ETc (mm/d)")
    ax1.set_title(f"Crop zone {cz}: {cprops['name']} (planting {cprops['planting_date']})")
    ax1.legend(loc="upper left"); ax2.legend(loc="upper right")
    plt.tight_layout(); plt.show()


## 19. Run the full spatiotemporal soil water balance

In [ ]:
spatiotemporal_df = calculate_spatiotemporal_optimized(
    weather_df, soil_gdf, crop_gdf, soil_configs, crop_configs,
    lat=LAT, h=ELEVATION_M, system_type=IRRIGATION_SYSTEM,
)

print(f"spatiotemporal_df: {len(spatiotemporal_df)} rows, "
      f"{spatiotemporal_df[['crop_zone','soil_zone']].drop_duplicates().shape[0]} unique units.")
print(f"Total recommended irrigation across all units: "
      f"{spatiotemporal_df['IRR'].sum():.0f} mm-events (sum, not per area).")
spatiotemporal_df.head()


## 20. Single-zone deep dive

Pick one (`crop_zone`, `soil_zone`) pair and inspect Dr, IRR, P, ETc.

In [ ]:
# >>> EDIT THESE TWO LINES to inspect a different unit <<<
INSPECT_CROP_ZONE = sorted(crop_gdf["crop_zone"].unique())[0]
INSPECT_SOIL_ZONE = sorted(soil_gdf["soil_zone"].unique())[0]

zd = spatiotemporal_df[
    (spatiotemporal_df["crop_zone"] == INSPECT_CROP_ZONE) &
    (spatiotemporal_df["soil_zone"] == INSPECT_SOIL_ZONE)
].copy()

if zd.empty:
    print(f"No data for crop {INSPECT_CROP_ZONE} x soil {INSPECT_SOIL_ZONE}.")
else:
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    axes[0].plot(zd["Date"], zd["Dr"], color="brown", label="Dr (deficit)")
    axes[0].plot(zd["Date"], zd["trigger_level"], "--", color="red", label="Trigger")
    axes[0].fill_between(zd["Date"], 0, zd["TAW"], color="lightyellow", alpha=0.4, label="TAW")
    axes[0].set_ylabel("mm"); axes[0].legend(loc="upper right")
    axes[0].set_title(f"Crop {INSPECT_CROP_ZONE} x Soil {INSPECT_SOIL_ZONE} -- "
                      f"system={'Pressurized' if IRRIGATION_SYSTEM==0 else 'Surface'}")
    axes[1].bar(zd["Date"], zd["P"],   color="steelblue", width=1.0, label="P (mm)")
    axes[1].bar(zd["Date"], zd["IRR"], color="forestgreen", width=1.0, alpha=0.8, label="IRR (mm)")
    axes[1].set_ylabel("mm/d"); axes[1].legend(loc="upper right")
    axes[2].plot(zd["Date"], zd["ETc"], color="orangered", label="ETc")
    axes[2].plot(zd["Date"], zd["ETo_used"], color="orange", alpha=0.6, label="ETo")
    axes[2].set_ylabel("mm/d"); axes[2].legend(loc="upper right")
    axes[2].set_xlabel("Date")
    plt.tight_layout(); plt.show()

    n_events = int((zd["IRR"] > 0).sum())
    print(f"Recommended irrigation events: {n_events}")
    print(f"Total recommended depth      : {zd['IRR'].sum():.0f} mm")


## 21. Recommended irrigation schedule

The cell below extracts only the days where `IRR > 0`, writes them to
`results/recommended_schedule.csv`, and shows the table. This is the file
you will edit to accept / reject / modify events in the next step.

In [ ]:
RESULTS_DIR = Path("/content/results") if IN_COLAB else Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

events = spatiotemporal_df[spatiotemporal_df["IRR"] > 0].copy()
events["Date"] = pd.to_datetime(events["Date"]).dt.date
sched = events[["Date","crop_zone","soil_zone","crop_type","IRR","Dr","ETc","P"]].copy()
sched = sched.rename(columns={"IRR":"IRR_recommended_mm","Dr":"Dr_mm","ETc":"ETc_mm","P":"P_mm"})
sched["decision"]  = "accept"   # one of: accept, reject, modify
sched["actual_mm"] = sched["IRR_recommended_mm"]   # used only when decision=='modify'

sched_path = RESULTS_DIR / "recommended_schedule.csv"
sched.to_csv(sched_path, index=False)
print(f"Wrote {len(sched)} recommended events -> {sched_path}")
sched.head(20)


### How to edit decisions

Open `results/recommended_schedule.csv` (Colab: double-click in the file
panel; VS Code: open in the editor). For each row, set `decision` to:

* `accept` — keep the recommended amount.
* `reject` — set the irrigation that day to zero.
* `modify` — set `actual_mm` to the depth you actually applied.

Save the file. Then run the next cell to recompute the soil water balance
with your decisions.

In [ ]:
def apply_decisions_and_recompute(spatiotemporal_df: pd.DataFrame,
                                  decisions_csv: Path) -> pd.DataFrame:
    """Read user decisions and recompute Dr / DP downstream of each event."""
    df = spatiotemporal_df.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    if not decisions_csv.exists():
        print(f"No decisions file at {decisions_csv}; returning input unchanged.")
        return df

    dec = pd.read_csv(decisions_csv)
    dec["Date"] = pd.to_datetime(dec["Date"])

    # 1. Override IRR per (date, crop_zone, soil_zone)
    for _, r in dec.iterrows():
        mask = ((df["Date"] == r["Date"]) &
                (df["crop_zone"].astype(str) == str(r["crop_zone"])) &
                (df["soil_zone"].astype(str) == str(r["soil_zone"])))
        if not mask.any():
            continue
        if r["decision"] == "reject":
            df.loc[mask, "IRR"] = 0.0
        elif r["decision"] == "modify":
            df.loc[mask, "IRR"] = float(r["actual_mm"])
        # accept -> leave as-is

    # 2. Recompute Dr, Dri, DP per (crop_type, soil_zone) using FAO-56 water balance
    df = df.sort_values(["crop_type","soil_zone","Date"]).reset_index(drop=True)
    for (ct, sz), grp in df.groupby(["crop_type","soil_zone"], sort=False):
        prev_def = 0.0
        taw = grp["TAW"].iloc[0]
        for idx, row in grp.iterrows():
            p_v = row["P"]; irr_v = row["IRR"]; etc_v = row["ETc"]
            dp = max(0.0, p_v + irr_v - etc_v - (taw - prev_def))
            new_def = max(0.0, min(prev_def - p_v - irr_v + etc_v + dp, taw))
            df.at[idx, "Dr"]  = new_def
            df.at[idx, "Dri"] = new_def
            df.at[idx, "DP"]  = dp
            prev_def = new_def
    return df

final_df = apply_decisions_and_recompute(spatiotemporal_df, sched_path)
final_path = RESULTS_DIR / "final_schedule_with_decisions.csv"
final_df.to_csv(final_path, index=False)
print(f"Wrote recomputed water balance -> {final_path}")

# Visualise: recommended (faded) vs actual (solid) for the inspected unit
zd2 = final_df[(final_df["crop_zone"] == INSPECT_CROP_ZONE) &
               (final_df["soil_zone"] == INSPECT_SOIL_ZONE)].copy()
zd_orig = spatiotemporal_df[(spatiotemporal_df["crop_zone"] == INSPECT_CROP_ZONE) &
                            (spatiotemporal_df["soil_zone"] == INSPECT_SOIL_ZONE)].copy()
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
axes[0].plot(zd_orig["Date"], zd_orig["Dr"], color="lightcoral", lw=1, label="Dr (recommended)")
axes[0].plot(zd2["Date"],     zd2["Dr"],     color="darkred",   lw=1.5, label="Dr (after decisions)")
axes[0].set_ylabel("Dr (mm)"); axes[0].legend()
axes[1].bar(zd_orig["Date"], zd_orig["IRR"], color="lightgreen", width=1.0, label="Recommended IRR")
axes[1].bar(zd2["Date"],     zd2["IRR"],     color="forestgreen", width=1.0, alpha=0.7, label="Actual IRR")
axes[1].set_ylabel("mm/d"); axes[1].legend()
plt.tight_layout(); plt.show()


## 22. Aggregations across all units

In [ ]:
agg = (final_df.groupby(["crop_zone","soil_zone","crop_type"])
       [["IRR","ETc","P","DP"]].sum().reset_index())
agg.columns = ["crop_zone","soil_zone","crop_type","IRR_sum_mm","ETc_sum_mm","P_sum_mm","DP_sum_mm"]
agg["irrigation_events"] = (final_df.groupby(["crop_zone","soil_zone","crop_type"])
                            ["IRR"].apply(lambda s: int((s>0).sum())).values)
print("Seasonal totals per management unit:")
display(agg) if "display" in globals() else print(agg)

# Stacked bar: P + IRR vs ETc per unit
fig, ax = plt.subplots(figsize=(10, 4.5))
units = agg["crop_zone"].astype(str) + "/" + agg["soil_zone"].astype(str)
ax.bar(units, agg["P_sum_mm"],   color="steelblue", label="P")
ax.bar(units, agg["IRR_sum_mm"], bottom=agg["P_sum_mm"], color="forestgreen", label="IRR")
ax.plot(units, agg["ETc_sum_mm"], "ko-", label="ETc")
ax.set_ylabel("Seasonal total (mm)"); ax.set_xlabel("crop_zone / soil_zone")
ax.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

# Heatmap: Dr over time per unit
pivot = final_df.assign(unit = final_df["crop_zone"].astype(str)+"/"+final_df["soil_zone"].astype(str))
pivot = pivot.pivot_table(index="unit", columns="Date", values="Dr", aggfunc="mean")
fig, ax = plt.subplots(figsize=(12, 0.4*len(pivot)+1.5))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax, cbar_kws={"label":"Dr (mm)"})
ax.set_title("Soil water deficit over time, per management unit")
plt.tight_layout(); plt.show()


## 23. Prescription map for a chosen date

In [ ]:
# >>> EDIT this date to any day in the season <<<
PRESCRIPTION_DATE = pd.to_datetime(today).normalize()

# Snap to nearest available date
all_dates = pd.to_datetime(final_df["Date"]).dt.normalize().unique()
nearest = pd.to_datetime(min(all_dates, key=lambda d: abs(d - PRESCRIPTION_DATE)))
print(f"Using nearest available date: {nearest.date()}")

snap = final_df[pd.to_datetime(final_df["Date"]).dt.normalize() == nearest][
    ["crop_zone","soil_zone","IRR","Dr"]
]
intersection_with_irr = intersection_gdf.merge(
    snap, on=["crop_zone","soil_zone"], how="left"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
intersection_with_irr.plot(column="IRR", ax=axes[0], legend=True,
                           cmap="Blues", edgecolor="black",
                           legend_kwds={"label":"IRR (mm)"})
axes[0].set_title(f"Recommended irrigation depth -- {nearest.date()}")
intersection_with_irr.plot(column="Dr", ax=axes[1], legend=True,
                           cmap="YlOrRd", edgecolor="black",
                           legend_kwds={"label":"Dr (mm)"})
axes[1].set_title(f"Soil water deficit -- {nearest.date()}")
for ax in axes:
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout()
fig.savefig(RESULTS_DIR / f"prescription_map_{nearest.date()}.png", dpi=150)
plt.show()
print(f"Saved -> {RESULTS_DIR/f'prescription_map_{nearest.date()}.png'}")


## 24. Validation — modeled `ETc` vs satellite-observed `AETI`

WaPOR AETI is a dekadal (10-day) total in mm/dekad. The cell aggregates
the modeled ETc to dekads and compares the two for each unit.

In [ ]:
if len(aeti_s) == 0:
    print("No AETI observations available -- skipping validation.")
else:
    # Aggregate modeled ETc to the AETI dekad endpoints
    aeti_df = aeti_s.reset_index()
    aeti_df.columns = ["Date","AETI_dekad"]
    aeti_df["Date"] = pd.to_datetime(aeti_df["Date"])
    val_rows = []
    for (cz, sz), grp in final_df.groupby(["crop_zone","soil_zone"], sort=False):
        g = grp.set_index(pd.to_datetime(grp["Date"]))[["ETc"]].copy()
        for _, dr in aeti_df.iterrows():
            d0 = dr["Date"]; d1 = d0 + pd.Timedelta(days=10)
            etc_dekad = g.loc[(g.index >= d0) & (g.index < d1), "ETc"].sum()
            if etc_dekad > 0:
                val_rows.append({"crop_zone":cz, "soil_zone":sz,
                                 "Date":d0, "ETc_model_mm":etc_dekad,
                                 "AETI_obs_mm":dr["AETI_dekad"]})
    val_df = pd.DataFrame(val_rows)

    fig, ax = plt.subplots(figsize=(10, 5))
    for (cz, sz), g in val_df.groupby(["crop_zone","soil_zone"]):
        ax.plot(g["Date"], g["ETc_model_mm"], "-",  label=f"{cz}/{sz} ETc (model)")
        ax.plot(g["Date"], g["AETI_obs_mm"],  "--", label=f"{cz}/{sz} AETI (WaPOR)")
    ax.set_ylabel("mm/dekad"); ax.legend(fontsize=8, ncol=2)
    ax.set_title("Modeled ETc vs satellite AETI (dekadal)")
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / "eto_aeti_validation.png", dpi=150)
    plt.show()


## 25. Export results

All artefacts go into `results/`. In Colab, the next cell triggers a
download of the main outputs to your machine.

In [ ]:
spatiotemporal_df.to_csv(RESULTS_DIR / "spatiotemporal_full.csv", index=False)
final_df.to_csv(RESULTS_DIR / "final_schedule_with_decisions.csv", index=False)
agg.to_excel(RESULTS_DIR / "seasonal_totals.xlsx", index=False)
print("Files written:")
for p in sorted(RESULTS_DIR.iterdir()):
    print("  ", p.name, f"({p.stat().st_size/1024:.1f} kB)")

if IN_COLAB:
    from google.colab import files
    for fname in ["recommended_schedule.csv",
                  "final_schedule_with_decisions.csv",
                  "spatiotemporal_full.csv",
                  "seasonal_totals.xlsx"]:
        try:
            files.download(str(RESULTS_DIR / fname))
        except Exception as exc:
            print(f"Skip download {fname}: {exc}")


## 26. Suggested experiments

Re-run only the cells indicated; the rest will pick up the new state.

| Experiment | Edit | Re-run from |
|---|---|---|
| Switch from Pressurized to Surface system | `IRRIGATION_SYSTEM = 1` (Cell 4) | Cell 19 |
| Change soil texture for one zone | `soil_configs[<zone>]["soil_type"]` (Cell 10) | Cell 19 |
| Test +2 °C climate scenario | `weather_df[["Tmax","Tmin"]] += 2.0` (new cell after 15) | Cell 16 |
| Postpone planting by 14 days | `crop_configs[<zone>]["planting_date"]` (Cell 10) | Cell 11 |
| Try L2 instead of L1 for AETI | `WAPOR_LEVEL = "L2"` (Cell 4) | Cell 13 |
| Reject every event in the second half of the season | edit `recommended_schedule.csv` | Cell 21 |

**Things to think about**

* When you reject an irrigation event, watch how `Dr` climbs and how the
  next recommended event arrives sooner *and* larger.
* Compare the seasonal totals table when you switch system type — surface
  irrigation usually has fewer but much larger events.
* Note where modeled `ETc` and WaPOR `AETI` diverge — that gap is often
  caused by water stress that the model captures but the climatological
  forecast cannot anticipate.
